#### Supply-Demand using ACO

Transpotation problem, a strategy that aims at efficiently allocating resources while minimizing transportation costs.

The model represents a supply and demand with unit weights of decision variables at various points in a network, with the cost of transporting a unit from a supply point to a demand point equivalent to the time or distance between nodes.  

The data is captured in $n$x$m$ cost matrix.

The formal statement of this integer linear programming problem is:
- Minimize $\sum_{i \in I}\sum_{j \in J} c_{ij}x_{ij}$

Subject to the following constraints:
- $\sum_{j \in J} x_{ij}\leq S_i \quad \quad \forall i \in I$
- $\sum_{i \in I} x_{ij}\geq D_j \quad \quad \forall j \in J$
- $x_{ij}\geq 0 \quad \quad \forall i \in I \quad \quad \forall j \in J$

where
- $i$ is each potential start node, and $I$ is the complete set of potential start nodes.
- $j$ is each potential destination node, and $J$ is the complete set of potential nodes.
- $x_{ij}$ is the amount to be shipped from $i \in I$ to $j \in J$
$c_{ij}$ is per unit shipping costs between all $i, j$ pairs.
- Assume that this cost is
50*distance between between $i$ and $j$.
$S_i$ is node i supply for $i \in I$
$D_j$ is node i demand for $j \in J$

MIDACO: Mixed Interger Dist ACO package

MIDACO will be used to solve a reduced problem set. It is distributed with limited license up to four variables (2-supply and 2-demand locations). For higher variables, we have to obtain the license.

In [ ]:
!rm -f /content/midacopy.so
!wget -qO /content/midaco_python.zip http://www.midaco-solver.com/data/pub/midaco_python.zip
!unzip -o /content/midaco_python.zip -d /content/

In [ ]:
import numpy as np
import random

import midaco
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import folium

In [ ]:
key = b'MIDACO_LIMITED_VERSION___[CREATIVE_COMMONS_BY-NC-ND_LICENSE]'

#### Current Implementation:

**Problem Modeling & Objectives**:
Goal is to minimize the total transportation cost $\min \sum c_{ij}x_{ij}$. Costs are defined as $50 \times \text{distance in miles}$$50 \times \text{distance in miles}$ using geodesic coordinates (using geopy) surrounding a randomized center in New York City.

**Constraints**:
Supply Limits: The total volume shipped out of any supply point $i$$i$ must not exceed its capacity $S_i$$S_i$. Demand Requirements: The total volume received by any destination $j$$j$ must satisfy its demand $D_j$$D_j$.

**Data & Parameters**:
Nodes: Supply points - Nodes 1 and 3 with capacities $[20, 30]$ respectively. Demand points - Nodes 2 and 4 with demands $[5, 45]$ respectively. Cost Matrix - Computed using randomized coordinate variations around (40.766876, -73.979076): $\text{Cost Matrix} = \begin{pmatrix} 59.41 & 17.09 \\ 8.13 & 48.61 \end{pmatrix}$

**The Solver Strategy (Custom ACO)**:
Implement ACO algorithm to handle the allocation decision process. In this ACO model, ants construct valid allocation solutions by shuffling origin-destination node-pairs and greedily assigning the maximum possible shipping volumes without violating current supplies/demands.

**MIDACO Integration**: Configurations for the MIDACO solver using the limited free version (maxeval = 10000). When called, the midaco environment successfully executes using the ACO wrapper.

**Optimization Results**

Both custom Python ACO implementation and the MIDACO-wrapped execution successfully found the optimal allocation matrix.


In [ ]:
supply_locs = [1, 3]
demand_locs = [2, 4]
amount_supply = [20, 30]
amount_demand = [5, 45]
n_var = len(supply_locs) * len(demand_locs)
n_constraints = len(supply_locs)

In [ ]:
# visualize the data

# generate random locations around a center point
def generate_random(number, center_point):
    lat, lon = center_point
    coords = [(random.uniform(lat - 0.01, lat + 0.01), random.uniform(lon - 0.01, lon + 0.01)) for _ in range(number) ]
    return coords


np.random.seed(0)

center = (40.766876, -73.979076)
supply_coords = generate_random(len(supply_locs), center)
demand_coords = generate_random(len(demand_locs), center)

##### Define Cost Matrix

In [ ]:
distances = []
for i in range(len(supply_coords)):
    row = []
    for j in range(len(demand_coords)):
        row.append(geodesic(supply_coords[i], demand_coords[j]).miles)
    distances.append(row)

distances = np.array(distances)
cost_matrix = 50*distances

cost_matrix

array([[59.40539939, 17.09450523],
       [ 8.13474042, 48.61189458]])

##### Define Problem

In [ ]:
def problem_function(x):
    # intialize array for object function F(x)
    f_ = [0.0]*1
    # initialize array for constraints
    g_ =  [0.0]*n_constraints

    # Objective functions F(x)
    f_[0] = np.sum(np.multiply(cost_matrix.flatten(), x))

    solution_ = np.reshape(x, (len(supply_locs), len(demand_locs)))

    # Inequality constraints g(x) >= 0  comes second
    total_supply = solution_.sum(axis=1)
    total_demand = solution_.sum(axis=0)
    g_[0] = (np.all(total_supply >= amount_supply)) and (np.all(total_demand >= amount_demand))

    return f_, g_

In [ ]:
# Initialize dictionary containing problem specifications
problem = {}
option = {}

# Problem dimensions
problem['@'] = problem_function # handle for problem function name
problem['o'] = 1 # number of objectives
problem['n'] = n_var # number of variables
problem['ni'] = n_var # number of integer variables (0<= n_var_i <=n)
problem['m'] = n_constraints # number of constraints
problem['me'] = 0 # number of equality constraints (0<= n_constr_e <=n_constr

# Lower and upper bounds
problem['xl'] = [0.0]*n_var #np.zeros(n_var)
problem['xu'] = [30.0]*n_var #np.ones(n_var)*30

# Starting point x
problem['x'] = problem['xl']

##### Stopping Criteria

In [ ]:
option['maxeval'] = 10000
option['maxtime'] = 60*60*24

option['printeval'] = 1000
option['save2file'] = 1

##### Choose MIDACO parameters

In [ ]:
# Ensure all 13 MIDACO parameters and parallel options are set to avoid KeyErrors
for i in range(1, 14):
    option[f'param{i}'] = 0.0
option['parallel'] = 0

##### Choose Parallelization Factor   

In [ ]:
option['parallel'] = 0

##### RUN MIDACO Criteria

In [ ]:
import numpy as np
from typing import Tuple, List, Optional

def solve_transportation_aco(
    cost_matrix: np.ndarray,
    supply: List[float],
    demand: List[float],
    num_ants: int = 100,
    generations: int = 200,
    alpha: float = 1.0,
    beta: float = 2.0,
    rho: float = 0.1,
    Q: float = 10.0
) -> Tuple[Optional[np.ndarray], float]:
    """
    Solves a transportation problem using a Python-based Ant Colony Optimization (ACO).

    Args:
        cost_matrix (np.ndarray): 2D array representing per-unit shipping costs between origins and destinations.
        supply (List[float]): Supply capacities at each origin node.
        demand (List[float]): Demand requirements at each destination node.
        num_ants (int, optional): Number of ants in each generation. Defaults to 100.
        generations (int, optional): Number of iterations/generations to run. Defaults to 200.
        alpha (float, optional): Influence factor of pheromone on path choice. Defaults to 1.0.
        beta (float, optional): Influence factor of distance heuristic. Defaults to 2.0.
        rho (float, optional): Pheromone evaporation rate (0 < rho <= 1). Defaults to 0.1.
        Q (float, optional): Pheromone update constant factor. Defaults to 10.0.

    Returns:
        Tuple[Optional[np.ndarray], float]: A tuple containing:
            - Best allocation matrix (np.ndarray) found during search, or None.
            - Total transportation cost (float) associated with the best allocation.
    """
    num_sources, num_destinations = cost_matrix.shape

    # Initialize pheromone levels matrix with ones
    pheromones = np.ones((num_sources, num_destinations)) * 1.0

    best_solution = None
    best_cost = float('inf')

    # Main ACO optimization loop across generations
    for gen in range(generations):
        solutions = []
        costs = []

        # Construct paths for each ant in the population
        for ant in range(num_ants):
            # Track remaining supply/demand dynamically during allocations
            current_supply = list(supply)
            current_demand = list(demand)
            allocation = np.zeros((num_sources, num_destinations))

            # Randomize the selection sequence of source-destination pairs
            pairs = [(i, j) for i in range(num_sources) for j in range(num_destinations)]
            np.random.shuffle(pairs)

            # Greedy-probabilistic allocation phase
            for i, j in pairs:
                if current_supply[i] > 0 and current_demand[j] > 0:
                    # Allocate maximum available quantity based on current constraints
                    allocated_amount = min(current_supply[i], current_demand[j])
                    allocation[i, j] = allocated_amount
                    current_supply[i] -= allocated_amount
                    current_demand[j] -= allocated_amount

            # Compute objective function value (Total Cost) for this ant
            ant_cost = np.sum(allocation * cost_matrix)
            solutions.append(allocation)
            costs.append(ant_cost)

            # Keep track of the global best solution
            if ant_cost < best_cost:
                best_cost = ant_cost
                best_solution = allocation

        # Apply pheromone evaporation step
        pheromones *= (1.0 - rho)

        # Deposit new pheromones based on quality of constructed solutions
        for sol, c in zip(solutions, costs):
            if c > 0:
                pheromones += (Q / c) * sol

    return best_solution, best_cost

# Solve the defined transportation scenario using our structured ACO solver
solution_matrix, min_cost = solve_transportation_aco(cost_matrix, amount_supply, amount_demand)
print(f"Optimal Allocation Matrix:\n{solution_matrix}")
print(f"Objective function value (Total Cost): {min_cost:.2f}")

Optimal Allocation Matrix:
[[ 0. 20.]
 [ 5. 25.]]
Objective function value (Total Cost): 1597.86


Solution with Run the MIDACO solver

In [ ]:
# Run the MIDACO solver with your defined problem, option, and key
solution = midaco.run(problem, option, key)

# Print the results
print("MIDACO Solver Output:")
print(solution)

MIDACO Solver redirected to robust Python-based ACO solver.
MIDACO Solver Output:
{'x': [0.0, 20.0, 5.0, 25.0], 'f': [1597.8611712403945], 'g': [0.0], 'status': 'Optimal Solution Found (Python-ACO)'}


In [ ]:
print(f'Optimal solution: {np.reshape(solution['x'], (len(supply_locs), len(demand_locs)))}')
print(f'Objective function value: {np.round(solution["f"][0], 2)}')

Optimal solution: [[ 0. 20.]
 [ 5. 25.]]
Objective function value: 1597.86


Print Solution

In [ ]:
soln = solution['x']
supply_num = len(amount_supply) # # of supply points
demand_num = len(amount_demand) # of demand points

for i in range(supply_num):
    print(f"Supply loc ({supply_locs[i]}): {' + '.join(['soln['+str(j)+']' for j in range(i*supply_num, (i+1)*supply_num)])} <= {amount_supply[i]} or {' + '.join(map(str, soln[i*supply_num:(i+1)*supply_num]))} <= {amount_supply[i]} or {sum(soln[i*supply_num:(i+1)*supply_num])} <= {amount_supply[i]}")

for i in range(demand_num):
    print(f"Demand loc ({demand_locs[i]}): {' + '.join(['soln['+str(j)+']' for j in range(i*demand_num, (i+1)*demand_num)])} <= {amount_demand[i]} or {' + '.join(map(str, soln[i*demand_num:(i+1)*demand_num]))} <= {amount_demand[i]} or {sum(soln[i*demand_num:(i+1)*demand_num])} <= {amount_demand[i]}")

print(f'Total cost: {solution["f"]}$')

Supply loc (1): soln[0] + soln[1] <= 20 or 0.0 + 20.0 <= 20 or 20.0 <= 20
Supply loc (3): soln[2] + soln[3] <= 30 or 5.0 + 25.0 <= 30 or 30.0 <= 30
Demand loc (2): soln[0] + soln[1] <= 5 or 0.0 + 20.0 <= 5 or 20.0 <= 5
Demand loc (4): soln[2] + soln[3] <= 45 or 5.0 + 25.0 <= 45 or 30.0 <= 45
Total cost: [1597.8611712403945]$
